# Fine-Tuning Trendyol-LLM-7b-chat-v1.0 for Turkish NLI

**Supervised fine-tuning** of [Trendyol/Trendyol-LLM-7b-chat-v1.0](https://huggingface.co/Trendyol/Trendyol-LLM-7b-chat-v1.0) (Mistral-7B-based Turkish chat model) on the [yilmazzey/sdp2-nli](https://huggingface.co/datasets/yilmazzey/sdp2-nli) dataset for Natural Language Inference.

## Dataset
- **SNLI-TR** (`snli_tr_1_1`): ~549K train, ~9.8K val, ~9.8K test
- **MultiNLI-TR** (`multinli_tr_1_1`): ~392K train, ~9.8K val_matched, ~9.8K val_mismatched
- **TrGLUE MNLI** (`trglue_mnli`): ~165K train, ~9.2K val_matched/mismatched, ~9.2K test_matched/mismatched
- **Total**: ~1.1M training examples (3-class: entailment / neutral / contradiction)

## Motivation
Zero-shot evaluation of this model on sdp2-nli showed it **always predicts "neutral"** (~33% accuracy, 0.0 F1 on entailment and contradiction). Fine-tuning with LoRA on even a fraction of the dataset should yield dramatic improvements, especially on native Turkish TrGLUE examples.

## Hardware Requirements
- **Full precision**: ~28 GB VRAM (A100 40GB)
- **4-bit quantized + LoRA**: ~12–16 GB VRAM (T4/V100/RTX 4090)
- **With Unsloth**: ~10–14 GB VRAM (2× faster training, 60% less memory)
- Training time: ~2–4 hours on A100 for 50K examples, proportionally longer for full dataset

## Approach
1. 4-bit NF4 quantization via bitsandbytes
2. LoRA adapters on all attention + MLP projections
3. SFTTrainer from `trl` with Mistral `[INST]` chat template
4. Short-answer format (single label token) to avoid hallucinated reasoning
5. Evaluation on held-out TrGLUE test splits (native Turkish, not translated)

## 1. Install Dependencies

**Note**: If running on Colab, restart the runtime after installation.

In [ ]:
!pip install -q -U \
    transformers datasets peft trl accelerate \
    bitsandbytes scikit-learn tqdm \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Optional: flash-attn for faster attention (requires CUDA 11.8+ and compatible GPU)
# !pip install flash-attn --no-build-isolation

## 2. Imports & Device Setup

In [ ]:
import json
import os
import random
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Fine-tuning will be extremely slow on CPU.")

## 3. Constants

In [ ]:
REPO_ID = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
MODEL_ID = "Trendyol/Trendyol-LLM-7b-chat-v1.0"
NUM_LABELS = 3
LABEL_NAMES = ["entailment", "neutral", "contradiction"]

OUTPUT_DIR = "./trendyol-nli-finetuned"
RESULTS_DIR = "./results"
Path(RESULTS_DIR).mkdir(exist_ok=True)

# Training hyperparameters
MAX_SEQ_LENGTH = 1024
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_RATIO = 0.03

# Debug mode: set to True to use a small subset for quick iteration
DEBUG_MODE = True
DEBUG_TRAIN_SIZE = 10_000
DEBUG_EVAL_SIZE = 1_000

print(f"Model: {MODEL_ID}")
print(f"Dataset: {REPO_ID}")
print(f"Debug mode: {DEBUG_MODE} (train={DEBUG_TRAIN_SIZE}, eval={DEBUG_EVAL_SIZE})")

## 4. Load & Prepare Dataset

We combine training splits from all three configs into a single training set. For evaluation, we use:
- `trglue_mnli/test_matched` — native Turkish (most important)
- `trglue_mnli/test_mismatched` — native Turkish, genre-mismatched
- `snli_tr_1_1/test` — translated Turkish

**Warning**: The full training set is ~1.1M examples. Start with `DEBUG_MODE = True` (10K samples) to verify the pipeline works, then scale up.

In [ ]:
print("Loading datasets...")

train_splits = []
for cfg in CONFIGS:
    ds = load_dataset(REPO_ID, cfg)
    if "train" in ds:
        train_splits.append(ds["train"])
        print(f"  {cfg}/train: {len(ds['train']):,} examples")

train_dataset = concatenate_datasets(train_splits)
print(f"\nCombined training set: {len(train_dataset):,} examples")

# Evaluation splits (native Turkish TrGLUE is most important)
eval_ds_trglue = load_dataset(REPO_ID, "trglue_mnli")
eval_ds_snli = load_dataset(REPO_ID, "snli_tr_1_1")
eval_ds_mnli = load_dataset(REPO_ID, "multinli_tr_1_1")

eval_datasets = {
    "trglue_test_matched": eval_ds_trglue["test_matched"],
    "trglue_test_mismatched": eval_ds_trglue["test_mismatched"],
    "trglue_val_matched": eval_ds_trglue["validation_matched"],
    "snli_test": eval_ds_snli["test"],
    "multinli_val_matched": eval_ds_mnli["validation_matched"],
}

for name, ds in eval_datasets.items():
    print(f"  eval/{name}: {len(ds):,} examples")

In [ ]:
if DEBUG_MODE:
    train_dataset = train_dataset.shuffle(seed=SEED).select(range(min(DEBUG_TRAIN_SIZE, len(train_dataset))))
    eval_datasets = {
        k: v.shuffle(seed=SEED).select(range(min(DEBUG_EVAL_SIZE, len(v))))
        for k, v in eval_datasets.items()
    }
    print(f"DEBUG: Using {len(train_dataset):,} train, {sum(len(v) for v in eval_datasets.values()):,} eval examples")
else:
    train_dataset = train_dataset.shuffle(seed=SEED)
    print(f"FULL: Using {len(train_dataset):,} train examples")

### 4.1 Label Distribution Check

Verify labels are roughly balanced (~33% each). Imbalanced labels could bias the model.

In [ ]:
label_counts = Counter(train_dataset["label"])
total = sum(label_counts.values())
print("Training label distribution:")
for label_id in sorted(label_counts.keys()):
    name = LABEL_NAMES[label_id] if label_id < len(LABEL_NAMES) else f"unknown({label_id})"
    count = label_counts[label_id]
    print(f"  {name} ({label_id}): {count:>8,} ({count/total:.1%})")

if HAS_PLOT:
    fig, ax = plt.subplots(figsize=(6, 3))
    labels = [LABEL_NAMES[i] for i in sorted(label_counts.keys())]
    counts = [label_counts[i] for i in sorted(label_counts.keys())]
    ax.bar(labels, counts, color=["#2ecc71", "#3498db", "#e74c3c"])
    ax.set_title("Training Label Distribution")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

## 5. NLI Prompt Template

Trendyol-LLM-7b-chat-v1.0 is based on Mistral-7B and uses the `[INST]` chat template.

**Design decisions:**
- Turkish task description to align with the model's Turkish capability
- Force single-word answer (no reasoning) to avoid hallucination and simplify parsing
- Keep prompts short to maximize batch efficiency

**Pitfall**: The zero-shot baseline used a similar template but the model always predicted "neutral". Fine-tuning teaches the model to actually discriminate between classes.

In [ ]:
LABEL_ID_TO_TEXT = {0: "Entailment", 1: "Neutral", 2: "Contradiction"}
LABEL_TEXT_TO_ID = {v.lower(): k for k, v in LABEL_ID_TO_TEXT.items()}

SYSTEM_PROMPT = (
    "Sen bir Türkçe doğal dil çıkarım (NLI) uzmanısın. "
    "Verilen önerme ve hipotez arasındaki mantıksal ilişkiyi belirle. "
    "Sadece tek kelime cevap ver: Entailment, Neutral veya Contradiction."
)


def format_nli_prompt(premise: str, hypothesis: str) -> str:
    """Format premise/hypothesis into the Mistral [INST] chat template (no label)."""
    return (
        f"[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"Önerme: {premise}\n"
        f"Hipotez: {hypothesis}\n\n"
        f"Bu iki cümle arasındaki ilişki nedir? [/INST] "
    )


def format_nli_example(example: dict) -> dict:
    """Format a single NLI example as prompt + completion for SFT."""
    prompt = format_nli_prompt(example["premise"], example["hypothesis"])
    label_text = LABEL_ID_TO_TEXT[example["label"]]
    example["text"] = prompt + label_text
    return example


# Preview
sample = train_dataset[0]
formatted = format_nli_example(sample)
print("=== Example formatted input ===")
print(formatted["text"])
print(f"\n=== Token count (approx): {len(formatted['text'].split())} words ===")

In [ ]:
print("Formatting training data...")
train_dataset = train_dataset.map(format_nli_example, desc="Formatting train")

print("Formatting evaluation data...")
eval_formatted = {}
for name, ds in eval_datasets.items():
    eval_formatted[name] = ds.map(format_nli_example, desc=f"Formatting {name}")

# Use trglue_val_matched as the training-time eval split
eval_for_training = eval_formatted["trglue_val_matched"]

print(f"\nTrain examples: {len(train_dataset):,}")
print(f"Eval (during training): {len(eval_for_training):,}")

## 6. Model & Tokenizer Setup

### Quantization Strategy
- **4-bit NF4** with double quantization and bfloat16 compute dtype
- Reduces VRAM from ~28 GB to ~5–6 GB for the base model
- LoRA adapters add ~100–200 MB of trainable parameters

### LoRA Configuration
- `r=64`: Good balance between capacity and efficiency for a 7B model
- Target all linear projections in attention + MLP for maximum adaptation
- `lora_dropout=0.05`: Light regularization to prevent overfitting on translated data

### Pitfalls
- **Do NOT resize the tokenizer** — the original Trendyol tokenizer handles Turkish well
- **Gradient checkpointing** is essential to fit training in limited VRAM
- If you get OOM errors, reduce `LORA_R` to 32 or `MAX_SEQ_LENGTH` to 512

In [ ]:
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# Verify Turkish tokenization works properly
test_text = "Önerme ve hipotez arasındaki ilişkiyi belirle."
tokens = tokenizer.tokenize(test_text)
print(f"\nTurkish tokenization test: '{test_text}'")
print(f"  Tokens ({len(tokens)}): {tokens[:15]}{'...' if len(tokens) > 15 else ''}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {MODEL_ID} (4-bit quantized)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {model.num_parameters():,}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 7. Training Configuration

### Key hyperparameters
- **Learning rate**: `2e-4` — standard for LoRA fine-tuning of 7B models
- **Effective batch size**: `per_device_batch_size × gradient_accumulation = 2 × 8 = 16`
- **Epochs**: 1 for the full dataset (sufficient for >100K examples), 2–3 for debug subset
- **Cosine schedule** with warmup to avoid early instability
- **bf16** for memory-efficient mixed-precision training

### Overfitting safeguards
- SNLI-TR and MultiNLI-TR are machine-translated → model may overfit on translationese
- TrGLUE MNLI is native Turkish → monitor its eval metrics separately
- Stop early if trglue_val loss diverges from training loss

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",

    # Batch & accumulation
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,

    # Precision & memory
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",

    # Logging & evaluation
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,

    # Saving
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Misc
    seed=SEED,
    report_to="none",
    packing=False,
)

print("Training configuration:")
eff_batch = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = len(train_dataset) // eff_batch
print(f"  Effective batch size: {eff_batch}")
print(f"  Steps per epoch: ~{steps_per_epoch:,}")
print(f"  Total training steps: ~{steps_per_epoch * NUM_EPOCHS:,}")
print(f"  Warmup steps: ~{int(steps_per_epoch * NUM_EPOCHS * WARMUP_RATIO):,}")

## 8. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_for_training,
    processing_class=tokenizer,
)

print(f"Trainer ready. Starting training on {len(train_dataset):,} examples...")

In [ ]:
train_result = trainer.train()

print("\n=== Training Complete ===")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Training runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"  Samples/second: {train_result.metrics['train_samples_per_second']:.1f}")

## 9. Save Model

Save the LoRA adapters first (small, ~200 MB), then optionally merge into a full model.

In [ ]:
lora_output_dir = os.path.join(OUTPUT_DIR, "lora-adapters")
trainer.save_model(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"LoRA adapters saved to: {lora_output_dir}")

# Save training metrics
metrics_path = os.path.join(RESULTS_DIR, "training_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(train_result.metrics, f, indent=2)
print(f"Training metrics saved to: {metrics_path}")

In [ ]:
# Optional: Merge LoRA weights into base model for standalone inference
# This creates a full-size model (~14 GB in fp16) — skip if storage is limited

MERGE_AND_SAVE = False  # Set to True to create a merged model

if MERGE_AND_SAVE:
    print("Merging LoRA adapters into base model...")
    merged_model = model.merge_and_unload()

    merged_output_dir = os.path.join(OUTPUT_DIR, "merged")
    merged_model.save_pretrained(merged_output_dir)
    tokenizer.save_pretrained(merged_output_dir)
    print(f"Merged model saved to: {merged_output_dir}")

    del merged_model
    torch.cuda.empty_cache()
else:
    print("Skipping merge. Use LoRA adapters from:", lora_output_dir)

## 10. Evaluation

Generate predictions on each eval split and compute accuracy + macro F1.

### Label parsing
The model is trained to output a single word: `Entailment`, `Neutral`, or `Contradiction`. We parse the first word of the generated text and map it back to a label ID. Fallback to `neutral` (1) if parsing fails.

In [ ]:
LABEL_WORD_TO_ID = {
    "entailment": 0, "neutral": 1, "contradiction": 2,
    # Turkish variants
    "içerme": 0, "tarafsız": 1, "nötr": 1, "çelişki": 2,
}


def parse_generated_label(generated_text: str, prompt: str) -> int:
    """Extract the predicted NLI label from generated text."""
    continuation = generated_text[len(prompt):].strip()
    first_word = re.split(r"[\s.,;:!?]+", continuation)[0].lower().strip()
    return LABEL_WORD_TO_ID.get(first_word, 1)


def evaluate_split(model, tokenizer, dataset, split_name, batch_size=4):
    """Run generation-based evaluation on a dataset split."""
    model.eval()
    all_preds = []
    all_labels = []
    parse_failures = 0

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval {split_name}"):
        batch = dataset[i : i + batch_size]
        prompts = [
            format_nli_prompt(p, h)
            for p, h in zip(batch["premise"], batch["hypothesis"])
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for gen_text, prompt, gold_label in zip(generated_texts, prompts, batch["label"]):
            pred = parse_generated_label(gen_text, prompt)
            all_preds.append(pred)
            all_labels.append(gold_label)

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return {
        "accuracy": round(acc, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_per_class": {
            LABEL_NAMES[i]: round(f, 4) for i, f in enumerate(f1_per_class)
        },
        "predictions": all_preds,
        "labels": all_labels,
    }

In [ ]:
print("Running evaluation on all splits...\n")

all_results = {}
for split_name, dataset in eval_datasets.items():
    result = evaluate_split(model, tokenizer, dataset, split_name, batch_size=4)
    all_results[split_name] = {
        "accuracy": result["accuracy"],
        "f1_macro": result["f1_macro"],
        "f1_per_class": result["f1_per_class"],
    }
    print(f"\n{'='*60}")
    print(f"{split_name}:")
    print(f"  Accuracy:  {result['accuracy']:.4f}")
    print(f"  F1 Macro:  {result['f1_macro']:.4f}")
    for cls_name, f1_val in result['f1_per_class'].items():
        print(f"  F1 {cls_name}: {f1_val:.4f}")

    # Confusion matrix
    if HAS_PLOT:
        cm = confusion_matrix(result["labels"], result["predictions"], labels=[0, 1, 2])
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix: {split_name}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Save evaluation metrics in the same format as base_results
metrics_output = {}
for split_name, result in all_results.items():
    parts = split_name.split("_", 1)
    config = parts[0]
    split = parts[1] if len(parts) > 1 else split_name
    if config not in metrics_output:
        metrics_output[config] = {}
    metrics_output[config][split] = result

metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_output, f, indent=2)
print(f"\nEvaluation metrics saved to: {metrics_path}")

## 11. Comparison with Zero-Shot Baseline

The zero-shot Trendyol-LLM-7b-chat-v1.0 results (from `base_results/`):

| Split | Accuracy | F1 Macro | F1 Entailment | F1 Neutral | F1 Contradiction |
|-------|----------|----------|---------------|------------|------------------|
| snli_tr / test | 0.3277 | 0.1645 | 0.0 | 0.4936 | 0.0 |
| multinli_tr / val_matched | 0.3184 | 0.1610 | 0.0 | 0.4830 | 0.0 |
| trglue / test_matched | 0.3484 | 0.1722 | 0.0 | 0.5167 | 0.0 |
| trglue / test_mismatched | 0.3302 | 0.1655 | 0.0 | 0.4964 | 0.0 |

The model predicted "neutral" for **every single example** in zero-shot mode.

In [ ]:
zero_shot_results = {
    "snli_test": {"accuracy": 0.3277, "f1_macro": 0.1645},
    "multinli_val_matched": {"accuracy": 0.3184, "f1_macro": 0.1610},
    "trglue_test_matched": {"accuracy": 0.3484, "f1_macro": 0.1722},
    "trglue_test_mismatched": {"accuracy": 0.3302, "f1_macro": 0.1655},
}

print("\n" + "="*70)
print("COMPARISON: Zero-Shot vs Fine-Tuned")
print("="*70)
print(f"{'Split':<30} {'Zero-Shot Acc':>14} {'FT Acc':>10} {'FT F1':>10}")
print("-"*70)
for split_name in all_results:
    zs = zero_shot_results.get(split_name, {})
    ft = all_results[split_name]
    zs_acc = zs.get("accuracy", "N/A")
    ft_acc = ft["accuracy"]
    ft_f1 = ft["f1_macro"]
    improvement = ""
    if isinstance(zs_acc, float):
        improvement = f" (+{ft_acc - zs_acc:.4f})"
        zs_acc = f"{zs_acc:.4f}"
    print(f"{split_name:<30} {zs_acc:>14} {ft_acc:>10.4f}{improvement} {ft_f1:>10.4f}")

## 12. Interactive Inference

Test the fine-tuned model on custom examples.

In [ ]:
test_examples = [
    {
        "premise": "Bir adam parkta köpeğini gezdiriyor.",
        "hypothesis": "Bir adam dışarıda.",
        "expected": "Entailment",
    },
    {
        "premise": "Çocuklar okulda ders çalışıyor.",
        "hypothesis": "Çocuklar tatilde.",
        "expected": "Contradiction",
    },
    {
        "premise": "Kadın marketten alışveriş yapıyor.",
        "hypothesis": "Kadın akşam yemeği için malzeme alıyor.",
        "expected": "Neutral",
    },
]

model.eval()
print("Interactive Inference Examples\n")

for ex in test_examples:
    prompt = format_nli_prompt(ex["premise"], ex["hypothesis"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    pred_label = parse_generated_label(generated, prompt)
    pred_text = LABEL_ID_TO_TEXT[pred_label]

    status = "OK" if pred_text == ex["expected"] else "WRONG"
    print(f"Premise:    {ex['premise']}")
    print(f"Hypothesis: {ex['hypothesis']}")
    print(f"Expected:   {ex['expected']}")
    print(f"Predicted:  {pred_text} [{status}]")
    print()

## 13. Loading the Fine-Tuned Model (for later use)

To load the LoRA-adapted model in a new session:

In [ ]:
# # Uncomment to reload the fine-tuned model in a new session
#
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# MODEL_ID = "Trendyol/Trendyol-LLM-7b-chat-v1.0"
# LORA_PATH = "./trendyol-nli-finetuned/lora-adapters"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, device_map="auto",
#     trust_remote_code=True,
# )
# model = PeftModel.from_pretrained(base_model, LORA_PATH)
# tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
#
# print("Fine-tuned model loaded successfully.")

## Expected Results & Next Steps

### Expected improvements
- **Zero-shot baseline**: ~33% accuracy (always predicts "neutral")
- **After fine-tuning** (debug, 10K examples): ~55–70% accuracy, ~50–65% F1 macro
- **After fine-tuning** (full, 1M+ examples): ~75–85% accuracy, ~74–84% F1 macro
- Biggest gains expected on **entailment** and **contradiction** (from 0% F1 to >70%)

### Comparison targets (from base_results)
| Model | TrGLUE test_matched Acc | TrGLUE test_matched F1 |
|-------|------------------------|------------------------|
| mDeBERTa-v3-base-mnli-xnli | ~0.82 | ~0.82 |
| Qwen2-7B-Instruct (zero-shot) | ~0.55 | ~0.52 |
| Gemma-3-27b-it (zero-shot) | ~0.68 | ~0.67 |
| **Trendyol-7b (zero-shot)** | **0.35** | **0.17** |
| **Trendyol-7b (fine-tuned)** | **TBD** | **TBD** |

### Next steps
1. **Scale up**: Set `DEBUG_MODE = False` and train on the full dataset for 1 epoch
2. **Hyperparameter sweep**: Try `lr` in {1e-4, 2e-4, 5e-4}, `r` in {32, 64, 128}
3. **Prompt variants**: Test reasoning-style prompts ("Kısa açıklama yaz, sonra etiketle")
4. **Ensemble**: Combine predictions with mDeBERTa and Gemma for best overall accuracy
5. **Push to Hub**: Upload LoRA adapters to HuggingFace for reproducibility
6. **Update dashboard**: Wire fine-tune results into the Streamlit app's "Fine tune results" tab
7. **Error analysis**: Compare errors with BERT-based models on TrGLUE (model_distillation patterns)
8. **Cross-validation on native Turkish**: TrGLUE results are most meaningful since SNLI-TR/MultiNLI-TR are translated